# Scraping de paginas web: listado de URLs utilizadas

**Elaborado por:**

- ALEJANDRO DANTE NAVA CAMPO - 202245606
- JONATHAN TIRO CUANENEMI - 2022688865

Dado una pagina web, este notebook lista todas las URLs que utiliza (enlaces, imagenes, hojas de estilo, scripts, formularios, iframes, recursos con srcset, metadatos con URL). Es valido para cualquier sitio web: basta con ingresar la direccion en el campo correspondiente de la interfaz.

**Enfoque tecnico:**

1. El usuario ingresa una URL en un campo de texto, dentro de una interfaz grafica construida con ipywidgets.
2. Se descarga el HTML de la pagina con la libreria requests (con manejo de errores, timeout y cabecera User-Agent).
3. Se parsea el HTML con BeautifulSoup y se recorren las etiquetas que contienen URLs: a href, img src, link href, script src, form action, iframe src, source srcset y meta content (cuando apunta a una URL).
4. Cada URL relativa se convierte en absoluta con urllib.parse.urljoin.
5. Se eliminan duplicados y se clasifica cada URL por tipo (enlace, imagen, script, css, formulario, iframe, recurso responsive, metadato) y por alcance (interna si comparte dominio con la pagina original, externa si no).
6. El resultado se muestra en una tabla dentro de la propia interfaz grafica y puede exportarse a CSV.

**Contenido del notebook:**

- Preparacion: importacion de librerias.
- Logica de scraping: funciones de descarga, parseo y clasificacion.
- Interfaz grafica: campo de URL, botones de accion y tabla de resultados embebida.
- Limitaciones conocidas y ejemplo de salida esperada.


## Preparacion: importacion de librerias

In [1]:
# Instala dependencias si hiciera falta (descomentar si es necesario)
# %pip install requests beautifulsoup4 ipywidgets pandas

import requests                          # peticiones HTTP para descargar el HTML
from bs4 import BeautifulSoup            # parseo del arbol DOM del HTML descargado
from urllib.parse import urljoin, urlparse  # normalizacion de URLs relativas/absolutas
import pandas as pd                      # tabla de resultados (DataFrame) y exportacion a CSV
from datetime import datetime            # marca de tiempo para el nombre del CSV exportado

import ipywidgets as widgets             # componentes de la interfaz grafica (GUI)
from IPython.display import display, clear_output, HTML

print("Librerias cargadas correctamente.")


Librerias cargadas correctamente.


## Logica de scraping

Funciones principales:

- descargar_html(url): hace la peticion HTTP y devuelve el HTML como texto. Lanza una excepcion controlada si falla (timeout, 404, error de conexion, etc.).
- extraer_urls(html, base_url): recorre el DOM con BeautifulSoup y arma una lista de diccionarios {url, tipo, alcance} con todas las URLs encontradas, ya normalizadas a formato absoluto.
- clasificar_alcance(url, base_url): determina si una URL es interna (mismo dominio que la pagina analizada) o externa.
- valores_url_de_atributo(etiqueta, atributo, valor): traduce el valor "crudo" de un atributo HTML (que puede ser una sola URL o varias, como en srcset) a una lista de URLs individuales.

El listado de etiquetas es facilmente ampliable: ademas de a, img, link, script, iframe y form, se incluye soporte para source srcset (imagenes responsive, que pueden traer varias URLs separadas por comas) y meta content (metadatos que apuntan a una URL, como Open Graph o canonical). Para sumar una etiqueta nueva basta con agregar una entrada al diccionario ETIQUETAS_URL y, si su atributo no es una URL simple, un manejador especifico como el de srcset dentro de valores_url_de_atributo.


In [2]:
# Diccionario central: cada entrada indica que etiqueta HTML revisar y en que
# atributo suele venir la URL. Ampliar el scraper a nuevas etiquetas es tan
# simple como agregar una linea aqui (y, si el atributo no trae una URL
# directa, sumar un caso en valores_url_de_atributo).
ETIQUETAS_URL = {
    "a": "href",         # enlaces de navegacion
    "img": "src",        # imagenes
    "link": "href",      # hojas de estilo, favicons, preconnect, etc.
    "script": "src",     # scripts externos
    "iframe": "src",     # contenido embebido (videos, mapas, etc.)
    "form": "action",    # destino de los formularios
    "source": "srcset",  # variantes de imagen para diseno responsive
    "meta": "content",   # metadatos que a veces apuntan a una URL (og:image, canonical)
}

# Nombre legible para mostrar en la tabla de resultados, en vez del nombre
# tecnico de la etiqueta HTML.
TIPO_LEGIBLE = {
    "a": "enlace",
    "img": "imagen",
    "link": "recurso (css/link)",
    "script": "script",
    "iframe": "iframe",
    "form": "formulario",
    "source": "recurso responsive (srcset)",
    "meta": "metadato",
}


def descargar_html(url, timeout=10):
    """Descarga el HTML de una URL. Lanza excepcion si la peticion falla."""
    # Se define un User-Agent de navegador porque algunos sitios bloquean
    # o limitan las peticiones que llegan sin cabecera (por ejemplo, las
    # que hace la libreria requests por defecto).
    headers = {"User-Agent": "Mozilla/5.0 (compatible; NotebookScraper/1.0)"}
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()  # levanta una excepcion si el codigo de estado indica error (4xx, 5xx)
    return resp.text


def clasificar_alcance(url, base_url):
    """Devuelve 'interna' si la URL comparte dominio con base_url, 'externa' en caso contrario."""
    try:
        dominio_base = urlparse(base_url).netloc.lower()
        dominio_url = urlparse(url).netloc.lower()
        if not dominio_url:
            # Una URL sin dominio (ej. una ruta relativa ya resuelta) se
            # considera interna, porque necesariamente pertenece al mismo sitio.
            return "interna"
        # Se quita el prefijo 'www.' para no confundir ejemplo.com con www.ejemplo.com
        dominio_base = dominio_base.replace("www.", "")
        dominio_url = dominio_url.replace("www.", "")
        return "interna" if dominio_url == dominio_base else "externa"
    except Exception:
        return "desconocida"


def valores_url_de_atributo(etiqueta, atributo, valor):
    """Devuelve la lista de valores de URL contenidos en un atributo.

    La mayoria de los atributos (href, src, action) traen una unica URL.
    'srcset' puede traer varias URLs separadas por coma, cada una seguida
    de un descriptor de tamano (ej: 'img1.jpg 1x, img2.jpg 2x').
    'content' en meta solo se considera si su valor parece una URL.
    """
    valor = valor.strip()
    if not valor:
        return []

    if etiqueta == "source" and atributo == "srcset":
        # Un srcset tipico luce asi: "imgA.jpg 480w, imgB.jpg 800w".
        # Se separa por comas y, de cada fragmento, solo se toma la URL
        # (la primera palabra), descartando el descriptor de tamano/densidad.
        urls = []
        for parte in valor.split(","):
            candidato = parte.strip().split(" ")[0].strip()
            if candidato:
                urls.append(candidato)
        return urls

    if etiqueta == "meta" and atributo == "content":
        # No todo <meta content="..."> trae una URL (puede ser una
        # descripcion, un color, una fecha, etc.). Solo se conserva si el
        # valor luce como una direccion web o una ruta.
        if valor.startswith("http://") or valor.startswith("https://") or valor.startswith("//") or valor.startswith("/"):
            return [valor]
        return []

    # Caso general: el atributo trae una unica URL (href, src, action).
    return [valor]


def extraer_urls(html, base_url):
    """Extrae todas las URLs usadas por la pagina y devuelve un DataFrame."""
    soup = BeautifulSoup(html, "html.parser")
    filas = []
    vistos = set()  # evita filas duplicadas (misma URL + misma etiqueta)

    for etiqueta, atributo in ETIQUETAS_URL.items():
        for elemento in soup.find_all(etiqueta):
            valor_bruto = elemento.get(atributo)
            if not valor_bruto:
                continue  # la etiqueta existe pero no tiene el atributo esperado

            for valor in valores_url_de_atributo(etiqueta, atributo, valor_bruto):
                # Se descartan pseudo-URLs que no representan un recurso real
                if not valor or valor.startswith("javascript:") or valor.startswith("mailto:") or valor.startswith("#"):
                    continue
                # Convierte rutas relativas ("/img/logo.png") en absolutas
                # usando la URL de la pagina como base.
                url_absoluta = urljoin(base_url, valor)
                clave = (url_absoluta, etiqueta)
                if clave in vistos:
                    continue
                vistos.add(clave)
                filas.append({
                    "url": url_absoluta,
                    "tipo": TIPO_LEGIBLE[etiqueta],
                    "etiqueta_html": "<" + etiqueta + " " + atributo + ">",
                    "alcance": clasificar_alcance(url_absoluta, base_url),
                })

    df = pd.DataFrame(filas)
    if not df.empty:
        df = df.sort_values(["tipo", "url"]).reset_index(drop=True)
    return df


def scrapear_pagina(url):
    """Funcion de alto nivel: descarga + extrae. Punto de entrada usado por la interfaz grafica.

    Funciona con cualquier sitio web publico: solo se necesita su URL completa.
    """
    html = descargar_html(url)
    return extraer_urls(html, url)

print("Funciones del scraper definidas.")


Funciones del scraper definidas.


## Interfaz grafica

Componentes:

- Campo de texto: para ingresar la URL a analizar. Es valido para cualquier sitio web publico, no esta limitado a un dominio en particular.
- Boton "Scrapear": ejecuta el analisis y muestra la tabla de resultados dentro de la misma interfaz.
- Boton "Exportar CSV": guarda el ultimo resultado en un archivo resultado_scraping_<fecha>.csv.
- Area de resultados: tabla renderizada como HTML dentro de la interfaz, con resumen de totales por tipo y alcance.

Al ejecutar la celda se muestra la interfaz ya cargada con un analisis de ejemplo sobre https://es.wikipedia.org/wiki/Wikipedia. Se puede cambiar la URL por la de cualquier otro sitio y presionar "Scrapear" para analizarlo.


In [3]:
# --- Campo de entrada de URL --------------------------------------------
url_input = widgets.Text(
    value="https://es.wikipedia.org/wiki/Wikipedia",
    placeholder="https://ejemplo.com",
    description="URL:",
    layout=widgets.Layout(width="700px", height="45px"),
    style={"description_width": "60px"},
)
url_input.add_class("campo-url-grande")  # clase CSS propia para agrandar el texto interno

# --- Botones de accion ---------------------------------------------------
boton_scrapear = widgets.Button(
    description="Scrapear",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="220px", height="50px"),
)
boton_scrapear.add_class("boton-grande")

boton_exportar = widgets.Button(
    description="Exportar CSV",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="220px", height="50px"),
)
boton_exportar.add_class("boton-grande")

# --- Areas de salida dentro de la GUI ------------------------------------
resumen = widgets.HTML()          # resumen de totales (interna/externa)
tabla_resultado = widgets.HTML()  # tabla de resultados en formato HTML
mensaje_estado = widgets.HTML()   # mensajes de estado y errores

# Guarda el ultimo resultado en memoria para poder exportarlo a CSV
# sin tener que volver a scrapear la pagina.
ultimo_resultado = {"df": None, "url": None}


def renderizar_resultado(df, url):
    """Actualiza los widgets de resumen y tabla con el resultado del scraping."""
    total = len(df)
    internas = int((df["alcance"] == "interna").sum()) if total else 0
    externas = int((df["alcance"] == "externa").sum()) if total else 0

    partes = []
    partes.append("<div style=\"font-size:16px; margin-top:6px;\">")
    partes.append("<b>URL analizada:</b> " + url + "<br>")
    partes.append("<b>Total de URLs encontradas:</b> " + str(total) + " &nbsp; ")
    partes.append("<b>Internas:</b> " + str(internas) + " &nbsp; ")
    partes.append("<b>Externas:</b> " + str(externas))
    partes.append("</div>")
    resumen.value = "".join(partes)

    if df.empty:
        tabla_resultado.value = "<i>No se encontraron URLs en la pagina.</i>"
    else:
        # to_html genera directamente el markup de la tabla, que se asigna
        # al widget HTML para que se vea dentro de la propia interfaz.
        tabla_resultado.value = df.to_html(index=False, escape=True, classes="tabla-resultado")


def al_hacer_click_scrapear(b):
    """Manejador del boton Scrapear: valida la URL, ejecuta el scraping y
    muestra el resultado (o el error correspondiente) en la interfaz."""
    url = url_input.value.strip()
    if not url:
        mensaje_estado.value = "<span style=\"color:#b00; font-size:16px;\">Ingresa una URL valida.</span>"
        return

    mensaje_estado.value = "<span style=\"font-size:16px;\">Analizando " + url + " ...</span>"
    try:
        df = scrapear_pagina(url)
    except requests.exceptions.RequestException as e:
        # Errores de red: timeout, dominio inexistente, sin conexion, etc.
        mensaje_estado.value = "<span style=\"color:#b00; font-size:16px;\">Error al descargar la pagina: " + str(e) + "</span>"
        resumen.value = ""
        tabla_resultado.value = ""
        return
    except Exception as e:
        # Cualquier otro error inesperado (parseo, etc.) se muestra igual
        # dentro de la interfaz, sin interrumpir la ejecucion del notebook.
        mensaje_estado.value = "<span style=\"color:#b00; font-size:16px;\">Error inesperado: " + str(e) + "</span>"
        resumen.value = ""
        tabla_resultado.value = ""
        return

    ultimo_resultado["df"] = df
    ultimo_resultado["url"] = url

    mensaje_estado.value = "<span style=\"font-size:16px;\">Analisis completado.</span>"
    renderizar_resultado(df, url)


def al_hacer_click_exportar(b):
    """Manejador del boton Exportar CSV: guarda el ultimo resultado en disco."""
    df = ultimo_resultado["df"]
    if df is None or df.empty:
        mensaje_estado.value = "<span style=\"color:#b00; font-size:16px;\">Primero ejecuta un scraping antes de exportar.</span>"
        return
    nombre = "resultado_scraping_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv"
    df.to_csv(nombre, index=False, encoding="utf-8")
    mensaje_estado.value = "<span style=\"font-size:16px;\">Resultado exportado a: " + nombre + "</span>"


# Se conectan los botones con sus respectivos manejadores de evento.
boton_scrapear.on_click(al_hacer_click_scrapear)
boton_exportar.on_click(al_hacer_click_exportar)

# --- Estilos CSS de la interfaz (tabla y tipografia mas grande) ----------
_css_interfaz = (
    "<style>"
    ".tabla-resultado { border-collapse: collapse; width: 100%; font-size: 15px; } "
    ".tabla-resultado th, .tabla-resultado td { border: 1px solid #ddd; padding: 8px 12px; text-align: left; } "
    ".tabla-resultado th { background-color: #f2f2f2; } "
    ".campo-url-grande input { font-size: 16px !important; } "
    ".boton-grande { font-size: 16px !important; font-weight: bold; }"
    "</style>"
)
estilo_interfaz = widgets.HTML(_css_interfaz)

_titulo_interfaz = (
    "<h2>Scraper de URLs</h2>"
    "<p style=\"font-size:15px;\">Valido para cualquier sitio web: ingresa la direccion completa y presiona Scrapear.</p>"
)

# --- Composicion final de la interfaz (contenedor vertical) --------------
interfaz = widgets.VBox(
    [
        widgets.HTML(_titulo_interfaz),
        estilo_interfaz,
        widgets.HBox([url_input], layout=widgets.Layout(margin="0 0 10px 0")),
        widgets.HBox([boton_scrapear, boton_exportar], layout=widgets.Layout(margin="0 0 15px 0")),
        mensaje_estado,
        resumen,
        tabla_resultado,
    ],
    layout=widgets.Layout(padding="15px", border="1px solid #ccc", width="100%"),
)

display(interfaz)

# Carga inicial de ejemplo: deja la interfaz ya poblada con resultados
# apenas se ejecuta la celda, sin necesidad de hacer clic manualmente.
al_hacer_click_scrapear(None)


## Limitaciones conocidas

- El scraper analiza unicamente el HTML que entrega el servidor en la respuesta inicial; no ejecuta JavaScript. Si un sitio genera enlaces o imagenes de forma dinamica (por ejemplo, cargando contenido con JavaScript despues de que la pagina termino de cargar), esas URLs no apareceran en el resultado.
- No se consulta el archivo robots.txt del sitio ni se aplican limites de velocidad entre peticiones; el uso recomendado es para analisis puntuales, no para rastreo masivo o continuo de un sitio.
- Los enlaces tipo mailto:, javascript: y anclas internas (#seccion) se descartan intencionalmente porque no representan un recurso web navegable.
- La clasificacion de "interna" y "externa" se basa unicamente en el dominio (quitando el prefijo www.); subdominios distintos (por ejemplo, blog.ejemplo.com frente a ejemplo.com) se consideran externos entre si.
- Sitios que requieren inicio de sesion, o que bloquean peticiones automatizadas, pueden devolver un HTML incompleto o un error, lo cual se refleja en el mensaje de error mostrado por la interfaz.

## Ejemplo de salida esperada

Al analizar una pagina con el boton "Scrapear", la interfaz muestra primero un resumen similar a este:

URL analizada: https://es.wikipedia.org/wiki/Wikipedia
Total de URLs encontradas: 350 | Internas: 210 | Externas: 140

Debajo del resumen aparece la tabla de resultados, con una fila por cada URL encontrada y sus columnas url, tipo, etiqueta_html y alcance, por ejemplo:

| url | tipo | etiqueta_html | alcance |
|---|---|---|---|
| https://es.wikipedia.org/wiki/Wikipedia:Portada | enlace | `<a href>` | interna |
| https://es.wikipedia.org/static/images/icons/wikipedia.png | imagen | `<img src>` | interna |
| https://creativecommons.org/licenses/by-sa/4.0/ | enlace | `<a href>` | externa |
| https://es.wikipedia.org/w/load.php?... | script | `<script src>` | interna |

Los numeros y URLs exactos varian segun el sitio analizado y el momento en que se ejecute el scraping, ya que el contenido de las paginas web cambia con el tiempo.
